# Essentiality on A100 — smart cooccur + self-learning + thesis loop

End-to-end run for the cross-organism gene-essentiality project. Each section is a
toggle. Designed for a Colab **A100** runtime (Runtime → Change runtime type → A100).

**What this does**
1. PyTorch port of the AlphaFold-style ortholog-MSA model (GPU; leave-one-clade-out).
2. **Smart cooccurrence**: a calibrated stacker on the transformer's *abstention bucket* — maximises new coverage at a 90% precision floor (replaces the old hard ≥2-votes rule). Includes the **dN/dS** channel + ablation.
3. **Self-learning loop**: active-learning curves (random vs uncertainty vs low-brightness acquisition) — tells us whether more labels actually move the conditional residual.
4. **Thesis loop**: regenerates results + figures and splices them into `CAPSTONE_PAPER.md`.

**Data scope note.** The repo ships the curated ~48–91 labelled organisms; dN/dS is already baked into `af_msa_cache.npz`. There is **no separate 8,300-genome set** in the repo — that earlier "depth" feature was flat (−0.003 AUC) and its raw data isn't here. More genomes only help the *cooccur presence matrix*, and only if you've assigned them to the existing OG space (see `build_presence.py --extra_presence_csv`). Recommendation: run as-is first; expand presence only if you have OG-assigned extra genomes on Drive.

**Rough A100 timings**: transformer LOCO ~5–15 min (`--big` ~15–25), smart cooccur ~3–6 min, active-learning ~20–60 min (depends on clades×budgets), thesis loop <1 min. Full run ≈ **45–90 min**; skip active-learning for a ~20 min run.

## 0 · Runtime check + get the code

In [ ]:
import torch, subprocess, os
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
assert torch.cuda.is_available(), "Set Runtime → Change runtime type → A100 GPU"


In [ ]:
# Get the repo. Option A: private clone with a GitHub token. Option B: it's already at /content/cell.
import os
REPO_DIR = "/content/cell"
BRANCH   = "claude/vectorize-gex-propensity-NRqBW"
GH_TOKEN = os.environ.get("GH_TOKEN", "")   # set in Colab: os.environ['GH_TOKEN']='ghp_...'
if not os.path.isdir(REPO_DIR):
    url = (f"https://{GH_TOKEN}@github.com/nikku03/cell.git" if GH_TOKEN
           else "https://github.com/nikku03/cell.git")
    !git clone --branch $BRANCH --depth 1 $url $REPO_DIR
else:
    print("repo already present at", REPO_DIR)
%cd $REPO_DIR
!git rev-parse --abbrev-ref HEAD
# sanity: the only data we need
import os
for f in ["outputs/orphan/af_msa_cache.npz",
          "data/drive_import/labels/orthology_features.csv",
          "data/drive_import/labels/cooccurrence_features.csv"]:
    print(("OK  " if os.path.exists(f) else "MISSING "), f)


## 1 · Transformer (PyTorch, leave-one-clade-out)

`--big` uses multi-head + 2 blocks (worth it on the A100). Drop it for the parity single-head config.

In [ ]:
!python colab/af_torch.py --big
import json; print(json.dumps(json.load(open("outputs/orphan/af_torch_results.json"))["pooled"], indent=2))


## 2 · Smart cooccurrence rescue (calibrated stacker @ P≥0.90)

Optimises coverage at a fixed 90% precision floor; reports the dN/dS marginal too. To use an expanded presence matrix, first run `!python colab/build_presence.py --extra_presence_csv /content/drive/.../extra_og.csv` then pass `--presence_npz outputs/orphan/presence_matrix.npz`.

In [ ]:
!python colab/smart_cooccur.py
import json; r=json.load(open("outputs/orphan/smart_cooccur_results.json"))
for tag in ("with_dnds","no_dnds"):
    s=r[tag]["smart_combined"]; t=r[tag]["transformer"]; d=r[tag]["delta"]
    print(f"{tag:10s}  transformer {t['coverage']:.3f}@{t['precision']:.3f}  ->  "
          f"combined {s['coverage']:.3f}@{s['precision']:.3f}  ({d['coverage_pp']:+.2f}pp cov)")
print("dN/dS marginal coverage:", r["dnds_marginal_coverage_pp"], "pp")


In [ ]:
# coverage/precision tradeoff curve (smart cooccur, with dN/dS)
import json, matplotlib.pyplot as plt
cur=json.load(open("outputs/orphan/smart_cooccur_results.json"))["with_dnds"]["tradeoff_curve"]
plt.figure(figsize=(6,4))
plt.plot([c["combined_coverage"] for c in cur],[c["combined_precision"] for c in cur],"o-")
plt.axhline(0.90,ls="--",c="r",label="precision floor 0.90")
plt.xlabel("combined coverage"); plt.ylabel("combined precision"); plt.legend(); plt.grid(alpha=.3)
plt.title("Smart cooccur: coverage vs precision"); plt.show()


## 3 · Self-learning loop (active learning)

Steep curve ⇒ labels are cheap and AL works; flat ⇒ the residual is irreducibly conditional and we need condition-aware data, not more labels. Trim clades/budgets to shorten.

In [ ]:
!python colab/active_learning.py --clades ralstonia,shewanella --budgets 0,200,500,1000,2000
from IPython.display import Image
Image("outputs/orphan/active_learning.png")


## 4 · Thesis production loop

Regenerates `RESULTS_AUTOGEN.md` and splices the numbers/figures into `CAPSTONE_PAPER.md` between `<!--AUTOGEN_START-->`/`<!--AUTOGEN_END-->`.

In [ ]:
!python colab/thesis_loop.py
print(open("outputs/orphan/RESULTS_AUTOGEN.md").read())


## 5 - Deep trace: channel ablation (option 1)

Drops each cooccur channel in turn, refits the cross-clade stacker, and prints
the marginal coverage contribution at the 90% precision floor. Also dumps the
final stacker coefficients and shows 5 example rescued genes with every
channel score visible -- so you can see exactly how the decision was made.

~5 min on A100.

In [ ]:
!python colab/deep_trace.py 2>&1 | tee /content/deep_trace.log
print("\n=== machine-readable summary ===")
import json; print(json.dumps(json.load(open("/content/cell/outputs/orphan/deep_trace_ablation.json")), indent=2))

## 6 - AL budget sweep to 10k (option 2)

Pushes the active-learning budget out to 10,000 to find where the AUC curve
flattens. For each fit, prints which genes were revealed, delta AUC vs
baseline, and a sample of genes whose prediction flipped from unsure to
confident.

~30 min on A100 (6 budgets x ~3 min/fit on Ralstonia with uncertainty +
random).

In [ ]:
!python colab/al_sweep_deep.py --clades ralstonia --strategies uncertainty,random \
    --budgets 0,500,1000,2000,5000,10000 2>&1 | tee /content/al_sweep.log
from IPython.display import Image; Image("/content/cell/outputs/orphan/al_sweep_deep.png")

## 7 · Save results back

Commit to the working branch (needs a token with write access), or download the outputs.

In [ ]:
# Option A: commit + push (requires GH_TOKEN with write scope)
!git add outputs/orphan/af_torch_results.json outputs/orphan/af_torch_preds.npz \
         outputs/orphan/smart_cooccur_results.json outputs/orphan/active_learning_results.json \
         outputs/orphan/active_learning.png outputs/orphan/RESULTS_AUTOGEN.md CAPSTONE_PAPER.md
!git -c user.email="chhillarnaresh03@gmail.com" -c user.name="colab" commit -m "A100 run: transformer + smart cooccur + active learning + thesis loop" || echo "nothing to commit"
# !git push   # uncomment if cloned with a write token


In [ ]:
# Option B: download a zip of the results
import shutil; shutil.make_archive("/content/essentiality_results","zip","outputs/orphan")
from google.colab import files; files.download("/content/essentiality_results.zip")
